### YOLOv8

Das fertige NHCW-Array kann nun YOLO übergeben werden.

Das Ergebnis wird von dem Ausgabe-Layer gelesen. YOLOv8 liefert ein Array in dem Format (N, 84, 8400) zurück. N steht für die Anzahl an Bildern.

Auf der 2. Achse liefert YOLO 84-Attribute zu einer Erkennung zurück.
Diese sind [central-x, central-y, width, height, detection score für die 80-Coco Klassen].
Auf der 3. Achse sind die Detections. Das Array muss daher zuerst transponiert werden, sodass die Detections vor den Attributen dazu stehen.

YOLO liefer außerdem immer 8400 Erkennungen zurück. Dafür liegt der Confidence-Score, also wie sicher das Model sich ist, bei den meisten Coco-Klassen deutlich unter 50%


In [25]:
import numpy as np
import cv2 as cv

from letterbox import letterbox

image: np.ndarray = cv.imread('pictures/bicycle.jpg')
resized, transform = letterbox(image)
blob = cv.dnn.blobFromImage(
    image=resized, scalefactor=1 / 255, size=(640, 640),
    swapRB=True, crop=False
)

# Load YOLO
net = cv.dnn.readNet('models/yolov8n.onnx')

# Put the blob into the input layer and receive the result from the output layer.
net.setInput(blob)
result = net.forward(net.getUnconnectedOutLayersNames())[0]
print(f"Output shape: {result.shape}")

# Transpose the 2 axis
result = np.transpose(result[0])
print(f"Transformed shape: {result.shape}")

# Go through every detection and look how many detections Yolo found
class_confidences = result[:, 4:]
max_confidences = np.max(class_confidences, axis=1)

valid_confidences = max_confidences >= 0.5

print(f"{valid_confidences.sum()} valid detections (No NMS)")


Output shape: (1, 84, 8400)
Transformed shape: (8400, 84)
14 valid detections (No NMS)
